In [ ]:
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config.constants import PROJECT_ROOT

In [ ]:
import zipfile
from pathlib import Path

#split_zip = Path(PROJECT_ROOT) / "saved_data" / "folds_Sina" / "split_ids.zip"
split_zip = Path(PROJECT_ROOT) / "saved_data" / "folds" / "global_split_ids 2" 


def load_splits(path=split_zip):
    return {
        name: {int(line) for line in (path / f"{name}.txt").read_text().split()}
        for name in ["train", "tuning", "held_out"]
    }

splits = load_splits()
for k, v in splits.items():
    print(f"{k:9s} {len(v):>7,} subjects")

train_glo = splits["train"]
val_glo = splits["tuning"]
test_glo = splits["held_out"]


In [ ]:
import pickle

FOLDS_DIR = Path(PROJECT_ROOT) / "saved_data" / "folds"
DTB = Path(PROJECT_ROOT) / "saved_data" / "cohorts" / "DTB" / "new"


def check_folds(cohort_name, fold=0, seed=42, first_adm_only=True, splits=splits):
    train_glo, val_glo, test_glo = splits["train"], splits["tuning"], splits["held_out"]

    suffix = "_firstadm" if first_adm_only else ""
    f = FOLDS_DIR / cohort_name / f"seed_{seed}{suffix}" / f"fold_{fold}.pkl"
    train_loc, val_loc, test_loc = pickle.load(open(f, "rb"))
    train_subj, val_subj, test_subj = train_loc[:, 0], val_loc[:, 0], test_loc[:, 0]

    df = pd.read_csv(DTB / f"{cohort_name}.csv.gz")
    cohort_first_df = df.sort_values("admittime", kind="stable").drop_duplicates("subject_id", keep="first")
    subj = set(df.subject_id.unique())
    print(f"=== {cohort_name} === subjects={len(subj)} first_adms={len(cohort_first_df)}")

    glo = {k: set(v) & subj for k, v in [("train", train_glo), ("val", val_glo), ("test", test_glo)]}
    loc = {k: set(v) & subj for k, v in [("train", train_subj), ("val", val_subj), ("test", test_subj)]}

    for name, d in [("global", glo), ("local", loc)]:
        print(f"\n-- {name} --")
        print("  train&val ", d["train"] & d["val"])
        print("  train&test", d["train"] & d["test"])
        print("  test&val  ", d["test"] & d["val"])
        covered = len(d["train"]) + len(d["val"]) + len(d["test"])
        print(f"  covered {covered}/{len(subj)}")
        for k in ("train", "val", "test"):
            print(f"  {k:5s} {len(d[k]):>6,} {len(d[k]) / len(subj):.3f}")

    print("\n-- cross-leakage (global vs local) --")
    print("  glo_train & loc_test", glo["train"] & loc["test"])
    print("  glo_train & loc_val ", glo["train"] & loc["val"])
    print("  glo_val   & loc_test", glo["val"] & loc["test"])
    print("  glo_val   & loc_train", glo["val"] & loc["train"])

    pretrain_seen = glo["train"] | glo["val"]
    leaked = pretrain_seen & loc["test"]
    uncovered = subj - (glo["train"] | glo["val"] | glo["test"])
    ok = not leaked and not uncovered
    print(f"\n  leaked into loc_test: {len(leaked)}  uncovered by global: {len(uncovered)}")
    print(f"  VERDICT: {'OK' if ok else 'FAIL'}")

    return {"global": glo, "local": loc, "leaked": leaked, "uncovered": uncovered, "ok": ok}


In [ ]:
COHORTS = [
    "C25-K59",
    "C64-C78",
    "C91-D63",
    "E05-E89",
    "E78-G81",
    "I12-K65",
    "I73-J81",
    "J45-J38",
    "K44-K59",
    "O26-O34",
]
for cohort in COHORTS:
    res = check_folds(cohort)
